# Benchmark · Diarización de hablantes · Bloque 1

**Responsable:** …  
**Fecha:** …

> Comparativa de modelos de diarización sobre la rueda de prensa de referencia del BCE.

In [ ]:
# Configuración común: rutas del proyecto y config.yaml
from src.config import ROOT, EVENTS_DIR, HISTORY_DIR, RAW_DIR, load_config, event_dir

cfg = load_config()
ref_date = cfg["reference_event"]
ref_dir = event_dir(ref_date)
audio_path = RAW_DIR / f"{ref_date}.wav"
print(f"Evento de referencia: {ref_date}")
print(f"Audio: {audio_path}")

## 1. Objetivo

Elegir el modelo de diarización que mejor separa a los hablantes (presidenta, vicepresidente, periodistas) en las ruedas de prensa del BCE.

El modelo ganador se usa para asignar `speaker` y `section` a cada segmento de `transcript.csv`.

## 2. Candidatos

| Modelo | Framework | Licencia | Por qué se incluye |
|---|---|---|---|
| pyannote/speaker-diarization-3.1 | pyannote.audio | MIT | Referencia, mayor precisión en benchmarks públicos |
| NeMo MSDD | NVIDIA NeMo | Apache 2.0 | Alternativa sin requisito de licencia HF |

## 3. Datos de referencia

Ground truth manual: anotamos los hablantes reales de la rueda de referencia a partir de la transcripción oficial del BCE (que identifica quién habla) y los timestamps del ASR.

La transcripción oficial distingue claramente:
- **Presidenta** (introductory statement + respuestas)
- **Vicepresidente** (intervenciones complementarias)
- **Periodistas** (preguntas)

In [ ]:
# Cargar transcripción ASR como referencia temporal
import pandas as pd
transcript = pd.read_csv(ref_dir / "transcript.csv")
print(f"Segmentos de transcripción: {len(transcript)}")
print(transcript[["start", "end", "speaker"]].head(10))


## 4. Métrica

**DER (Diarization Error Rate):** porcentaje de tiempo con hablante mal asignado, calculado con `pyannote.metrics`.

DER = (Falsa alarma + Pérdida + Confusión) / Tiempo total anotado

Secundaria: **número de hablantes detectados** vs. reales (3: presidenta, vicepresidente, periodistas como grupo).

In [ ]:
# Evaluar pyannote 3.1
from src.audio.models.diarization_backends import crear_backend_diarization, liberar_memoria

backend = crear_backend_diarization("pyannote/speaker-diarization-3.1")
diar = backend.diarizar(audio_path)
print(f"Turnos detectados: {len(diar)}")
print(f"Hablantes únicos: {diar['speaker_id'].nunique()}")
print(diar.head(10))
liberar_memoria()


## 5. Resultados

Comparar DER y número de hablantes detectados por cada candidato.

In [ ]:
# Comparativa (cuando NeMo esté implementado)
candidatos = [
    "pyannote/speaker-diarization-3.1",
    # "nvidia/nemo-msdd",  # activar cuando se implemente
]

resultados = []
for modelo_id in candidatos:
    backend = crear_backend_diarization(modelo_id)
    diar = backend.diarizar(audio_path)
    n_speakers = diar["speaker_id"].nunique()
    # DER se calcularía con pyannote.metrics si hay anotación de referencia
    resultados.append({
        "modelo": modelo_id,
        "hablantes_detectados": n_speakers,
        "turnos": len(diar),
        "vram_gb": backend.memoria_gb(),
    })
    liberar_memoria()

import pandas as pd
df = pd.DataFrame(resultados)
print(df.to_markdown(index=False))


## 6. Latencia y memoria

Hardware utilizado, tiempo de proceso y VRAM.

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")


## 7. Decisión

Modelo elegido y argumento. Anotar el ganador en `config.yaml` bajo `models.diarization`.